# EDA: химические дескрипторы и биологическая активность (IC50, CC50, SI)

Это отдельный этап исследования перед построением моделей.  
Задача EDA — понять структуру данных, качество признаков и риски (пропуски, выбросы, мультиколлинеарность), чтобы затем осознанно выбрать подход к предобработке и моделированию.

**Важно для постановки задачи:**
- `SI` вычисляется на основе `IC50` и `CC50`, поэтому в задачах прогнозирования `SI` нельзя использовать `IC50`/`CC50` как признаки;
- в задачах прогнозирования `IC50`/`CC50` нельзя использовать `SI` (иначе будет утечка целевой переменной).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = Path('data') / 'Данные_для_курсовой_Классическое_МО.xlsx'
df_raw = pd.read_excel(DATA_PATH)

print(f'Исходный shape: {df_raw.shape}')
print(f'Первые 8 колонок: {df_raw.columns[:8].tolist()}')

## 1) Обзор структуры данных: размерность, типы, базовые проверки

In [ ]:
display(df_raw.head(3))

print('\nТипы данных:')
print(df_raw.dtypes.value_counts())

print('\nОбщее число колонок:', len(df_raw.columns))
print('Есть ли служебная колонка Unnamed: 0?', 'Unnamed: 0' in df_raw.columns)

**Вывод по структуре:**
- датасет содержит **1001 наблюдение и 214 колонок**;
- все признаки числовые (`int64` и `float64`), что удобно для классических моделей;
- колонка `Unnamed: 0` выглядит как технический индекс из Excel и не несет химического смысла — далее исключаем её из анализа и моделирования.


## 2) Анализ пропусков

In [ ]:
df = df_raw.copy()
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print('Всего пропусков:', int(df.isna().sum().sum()))
print('Колонок с пропусками:', missing.shape[0])

display(missing.to_frame('missing_count').head(15))

if not missing.empty:
    plt.figure(figsize=(11, 4))
    sns.barplot(x=missing.index, y=missing.values, color='#4C72B0')
    plt.xticks(rotation=75, ha='right')
    plt.ylabel('Число пропусков')
    plt.title('Пропуски по колонкам')
    plt.tight_layout()
    plt.show()

**Вывод по пропускам:**
- пропусков мало (единичные значения в ограниченном числе дескрипторов), массового провала качества данных нет;
- удалять строки целиком нерационально (каждая строка — отдельное соединение);
- для моделирования выбираем **медианную импутацию**, чтобы не искажать распределения и сохранить весь датасет.


## 3) Описательная статистика

In [ ]:
targets = ['IC50, mM', 'CC50, mM', 'SI']

desc_targets = df[targets].describe().T
display(desc_targets)

selected_features = ['MolWt', 'TPSA', 'MolLogP', 'qed', 'NumHDonors', 'NumHAcceptors']
selected_features = [c for c in selected_features if c in df.columns]

display(df[selected_features].describe().T)

**Вывод по статистике:**
- `IC50`, `CC50` и особенно `SI` имеют большой разброс между квартилями и максимумами;
- медианы существенно ниже средних значений, что указывает на правостороннюю асимметрию;
- это типично для биологических метрик: большинство соединений умеренные/слабые, а небольшая группа даёт экстремальные значения.


## 4) Распределения признаков (гистограммы)

In [ ]:
plot_cols = ['IC50, mM', 'CC50, mM', 'SI'] + selected_features
plot_cols = [c for c in plot_cols if c in df.columns]

ncols = 3
nrows = int(np.ceil(len(plot_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(plot_cols):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i], color='#55A868')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

**Интерпретация распределений:**
- у целевых переменных заметны длинные правые хвосты;
- часть молекулярных дескрипторов ближе к «компактным» распределениям, но многие также негауссовы;
- для линейных моделей это означает чувствительность к масштабу и хвостам, поэтому масштабирование и регуляризация обязательны;
- для ансамблевых деревьев несимметричные распределения менее критичны, что часто даёт им преимущество на таких данных.


## 5) Boxplot и диагностика выбросов

In [ ]:
box_cols = ['IC50, mM', 'CC50, mM', 'SI', 'MolWt', 'TPSA', 'MolLogP']
box_cols = [c for c in box_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.reshape(-1)

for i, col in enumerate(box_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='#C44E52')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

# Оценим долю IQR-выбросов для таргетов
for t in ['IC50, mM', 'CC50, mM', 'SI']:
    q1 = df[t].quantile(0.25)
    q3 = df[t].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outlier_share = ((df[t] < low) | (df[t] > high)).mean()
    print(f'{t}: доля IQR-выбросов = {outlier_share:.3f}')

**Вывод по выбросам:**
- выбросы действительно присутствуют, особенно по `SI`;
- в контексте drug discovery это может быть не «ошибка», а редкие, но ценные активные соединения;
- поэтому в базовой версии проекта **не удаляем выбросы жёстко**, чтобы не потерять потенциально информативные примеры;
- вместо этого используем устойчивые методы (деревья, бустинг) и аккуратную импутацию.


## 6) Корреляционный анализ (heatmap)

In [ ]:
corr_targets = df.corr(numeric_only=True)[['IC50, mM', 'CC50, mM', 'SI']].drop(index=['IC50, mM', 'CC50, mM', 'SI'])

# Возьмем по 6 наиболее связанных признаков на каждый таргет
top_features = set()
for t in ['IC50, mM', 'CC50, mM', 'SI']:
    top_features.update(corr_targets[t].abs().sort_values(ascending=False).head(6).index.tolist())

top_features = sorted(top_features)
heatmap_df = df[top_features + ['IC50, mM', 'CC50, mM', 'SI']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_df, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Корреляционная карта (топ-признаки + таргеты)')
plt.tight_layout()
plt.show()

display(corr_targets.abs().sort_values(by='SI', ascending=False).head(10))

**Интерпретация корреляций:**
- сильных линейных связей у большинства дескрипторов с таргетами немного, что объясняет ограниченность чисто линейного подхода;
- заметна разная «природа» таргетов: набор полезных признаков для `SI` не совпадает полностью с `IC50`/`CC50`;
- это аргумент в пользу отдельных моделей под каждый таргет, а не одной универсальной модели.


## 7) Анализ целевых переменных (IC50, CC50, SI)

In [ ]:
targets = ['IC50, mM', 'CC50, mM', 'SI']

target_stats = pd.DataFrame({
    'median': df[targets].median(),
    'mean': df[targets].mean(),
    'std': df[targets].std(),
    'skewness': df[targets].skew(),
    'p95': df[targets].quantile(0.95),
    'max': df[targets].max(),
})
display(target_stats)

# Классификационные пороги
thresholds = {
    'IC50_median': float(df['IC50, mM'].median()),
    'CC50_median': float(df['CC50, mM'].median()),
    'SI_median': float(df['SI'].median()),
    'SI_threshold_8': 8.0,
}
print(thresholds)

class_distribution = {
    'IC50 > median': (df['IC50, mM'] > thresholds['IC50_median']).mean(),
    'CC50 > median': (df['CC50, mM'] > thresholds['CC50_median']).mean(),
    'SI > median': (df['SI'] > thresholds['SI_median']).mean(),
    'SI > 8': (df['SI'] > 8).mean(),
}
print('\nДоля положительного класса:')
for k, v in class_distribution.items():
    print(f'{k}: {v:.3f}')

**Вывод по целевым переменным:**
- все три таргета асимметричны, `SI` — наиболее «тяжелохвостый»;
- задачи по медиане почти сбалансированы (около 50/50), а `SI > 8` уже более сложная и умеренно несбалансированная постановка;
- для `SI` разумно ожидать более скромное качество в регрессии и более сильный разброс метрик между моделями.


## 8) Решения по предобработке перед моделями

In [ ]:
preprocessing_plan = pd.DataFrame(
    {
        'Шаг': [
            'Удаление служебного признака',
            'Обработка пропусков',
            'Масштабирование',
            'Train/test split',
            'Fix random_state',
            'Anti-leakage правила',
        ],
        'Решение': [
            'Удалить `Unnamed: 0`',
            'SimpleImputer(strategy="median")',
            'StandardScaler (для линейных моделей)',
            'test_size=0.2',
            'random_state=42',
            'IC50/CC50 без SI; SI без IC50 и CC50',
        ],
    }
)

display(preprocessing_plan)

### Финальный вывод EDA

1. Данные в целом пригодны для моделирования: числовые дескрипторы, мало пропусков, понятные таргеты.  
2. Таргеты имеют выраженную асимметрию и выбросы, поэтому линейные модели полезны как baseline, но ансамбли деревьев ожидаемо сильнее.  
3. Критично соблюдать анти-утечку для задач с `SI`, иначе качество будет искусственно завышено.  
4. На следующем этапе корректно сравниваем модели с кросс-валидацией и подбором гиперпараметров.


## EDA: итоговая интерпретация и anti-leakage вывод

EDA показывает, что `IC50`, `CC50` и особенно `SI` имеют асимметричные распределения и выбросы. Выбросы не удаляются автоматически, потому что в drug discovery экстремальные значения могут соответствовать биологически важным соединениям.

Ключевой методологический вывод: `SI` является функцией `CC50` и `IC50`, поэтому endpoints и их производные нельзя использовать как признаки. Финальный pipeline применяет descriptor-only подход и удаляет target-like колонки до обучения. Это делает результаты classification и regression научно корректнее и предотвращает искусственно завышенный ROC-AUC.

